In [0]:
%sql
create or replace temp view vw_dim_seller as(
  with dedup as (
    select
       seller_id
      ,seller_name
      ,row_number() over (
        partition by 
           seller_id
        order by 
           sale_date desc
          ,ingestion_timestamp desc
      )                                                             as order_by_date
    from workspace.pj_sales.tb_sales_silver
  )
  select
     seller_id
    ,seller_name
    ,from_utc_timestamp(current_timestamp(), 'America/Sao_Paulo')   as valid_from
    ,null                                                           as valid_to
  from dedup
  where order_by_date = 1
)

In [0]:
%sql
create or replace temp view vw_dim_seller_merge as
select
  vs.*,
  'u' as action
from vw_dim_seller vs -- Aqui o registro serve para ser atualizado
union all
select
  vs.*,
  'i' as action
from vw_dim_seller vs -- Aqui o registro serve para ser inserido

In [0]:
%sql
merge into workspace.pj_sales.tb_dim_seller_gold ts
using vw_dim_seller_merge vsm
on  ts.seller_id = vsm.seller_id
and ts.valid_to is null
and vsm.action = 'u'
when matched and vsm.action = 'u' then
  update set
    ts.valid_to = vsm.valid_from
when not matched and vsm.action = 'i' then
  insert (
     seller_id
    ,seller_name
    ,valid_from
    ,valid_to
  )
  values (
     vsm.seller_id
    ,vsm.seller_name
    ,vsm.valid_from
    ,vsm.valid_to
  )
